In [1]:
import carla
import random 
import cv2
import time 
import numpy as np
import math 
from scipy.spatial import cKDTree

In [2]:
# Connect to the client
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
#client.load_world('Town04')
world = client.get_world()
carla_map = world.get_map()

print("Successfully connected to CARLA and loaded the map!")

settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

# Check the boolean flag
if settings.synchronous_mode:
    print("Currently running in: SYNCHRONOUS Mode (Client-driven)")
else:
    print("Currently running in: ASYNCHRONOUS Mode (Server-driven)")

Successfully connected to CARLA and loaded the map!
Currently running in: SYNCHRONOUS Mode (Client-driven)


In [3]:
# Load all waypoints in the map
waypoint_dist = 2.0  # Distance between waypoints
waypoints = carla_map.generate_waypoints(waypoint_dist)
print(f"Generated {len(waypoints)} waypoints in the map.")

data = []
for wp in waypoints:
    if wp.lane_type == carla.LaneType.Driving:
        data.append([
            wp.transform.location.x, 
            wp.transform.location.y, 
            wp.lane_width
        ])

# Convert the list of waypoints to a NumPy array
waypoint_array = np.array(data)
print("Converted driving waypoints coordinates and lane widths to NumPy array.")
print(waypoint_array[:5])  # Print the first 5 waypoints for verification

Generated 3126 waypoints in the map.
Converted driving waypoints coordinates and lane widths to NumPy array.
[[-428.00054932   46.38977051   30.        ]
 [-449.1416626    25.10471916   30.        ]
 [-426.58154297   44.98036194   30.        ]
 [-447.72265625   23.69531059   30.        ]
 [-425.16253662   43.570961     30.        ]]


In [4]:
# Build a KD-tree for efficient nearest neighbor search
waypoint_tree = cKDTree(waypoint_array[:, :2])  # Use only x and y for the tree
print("KD-tree built for waypoint coordinates.")

# Test with a dummy trajectory (replace this with your actual trajectory points)
dummy_trajectory = np.array([
    [150.5, 20.1], 
    [152.0, 21.5], 
    [154.5, 23.0]
])

# Query the tree with the entire batch of trajectory points simultaneously.
# k=1 means we only want the single closest waypoint for each point.
start_time = time.time()
distances, indices = waypoint_tree.query(dummy_trajectory, k=1)
end_time = time.time()
print(f"Batch query completed in {end_time - start_time:.6f} seconds.")
# 'indices' contains the row numbers of the closest waypoints in your original array.

closest_waypoints = waypoint_array[indices]

print("\n--- Query Results ---")
print(f"Trajectory points:\n{dummy_trajectory}")
print(f"Closest waypoint coordinates:\n{closest_waypoints}")
print(f"Distances to those waypoints:\n{distances}")

KD-tree built for waypoint coordinates.
Batch query completed in 0.001008 seconds.

--- Query Results ---
Trajectory points:
[[150.5  20.1]
 [152.   21.5]
 [154.5  23. ]]
Closest waypoint coordinates:
[[ 201.03674316 -176.03005981   30.        ]
 [ 202.92637634 -175.53936768   30.        ]
 [ 208.58583069 -174.03126526   30.        ]]
Distances to those waypoints:
[202.53632458 203.51414747 204.31983891]


In [5]:
def visualize_waypoints_and_trajectory(waypoints, trajectory, closest_waypoints):
    # Create a blank image
    img_size = 500
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)

    # Scale and translate coordinates for visualization
    def transform_coords(coords):
        return (coords - np.min(coords, axis=0)) * (img_size / (np.ptp(coords, axis=0) + 1e-5))

    waypoints_vis = transform_coords(waypoints[:, :2])
    trajectory_vis = transform_coords(trajectory)
    closest_waypoints_vis = transform_coords(closest_waypoints[:, :2])

    # Draw waypoints
    for wp in waypoints_vis:
        cv2.circle(img, (int(wp[0]), int(wp[1])), 3, (255, 255, 255), -1)

    # Draw trajectory points
    for point in trajectory_vis:
        cv2.circle(img, (int(point[0]), int(point[1])), 5, (0, 255, 0), -1)

    # Draw closest waypoints
    for wp in closest_waypoints_vis:
        cv2.circle(img, (int(wp[0]), int(wp[1])), 5, (0, 0, 255), -1)

    # Display the image
    cv2.imshow('Waypoints and Trajectory', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [6]:
def query_valid_trajectory(trajectory_points, waypoint_tree, waypoint_array):
    # Query the tree with the entire batch of trajectory points simultaneously.
    # k=1 means we only want the single closest waypoint for each point.
    distances, indices = waypoint_tree.query(trajectory_points, k=1)
    closest_waypoints = waypoint_array[indices]
    
    # Check if the car has gone off the lane
    lane_widths = closest_waypoints[:, 2]  # Assuming lane width is the third column
    off_road_mask = distances > (lane_widths / 2.0)
    if np.any(off_road_mask):
        impact_index = np.argmax(off_road_mask)
        valid_trajectory = trajectory_points[:impact_index]
        return valid_trajectory

    return trajectory_points  # All points are valid if none are off-road

In [7]:
def predict_trajectory_to_impact(vehicle, wheelbase, waypoint_array, waypoint_tree, turn_direction="right", steer_intensity=1.0, time_horizon=4.0, dt=0.1):
    """
    Predicts a trajectory and stops exactly when it hits a road boundary 
    using a pre-computed KD-Tree and offline numpy array.
    Returns the valid trajectory points (carla.Location) and the distance traveled to impact.
    """
    transform = vehicle.get_transform()
    velocity = vehicle.get_velocity()
    physics = vehicle.get_physics_control() 
    
    speed = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
    
    # Return empty if stationary
    if speed < 0.1:
        return -1, []

    steer_intensity = max(0.0, min(1.0, steer_intensity))
    max_steer_rad = math.radians(physics.wheels[0].max_steer_angle)
    applied_steer_rad = max_steer_rad * steer_intensity
    
    if turn_direction.lower() == "left":
        steer_angle = -applied_steer_rad
    else: 
        steer_angle = applied_steer_rad

    x = transform.location.x
    y = transform.location.y
    z = transform.location.z 
    current_yaw = math.radians(transform.rotation.yaw)
    
    yaw_rate = (speed / wheelbase) * math.tan(steer_angle) if wheelbase > 0 else 0
    
    # Initialize variables
    trajectory_locations = []
    trajectory_coords = []  # To store [x, y] for the KD-Tree query
    
    accumulated_yaw = 0.0
    target_yaw_change = math.pi / 2.0  # 90 degrees max turn
    
    steps = int(time_horizon / dt)
    
    # 1. Generate full kinematics trajectory
    for _ in range(steps):
        if abs(accumulated_yaw) < target_yaw_change:
            active_yaw_rate = yaw_rate
        else:
            active_yaw_rate = 0.0 
            
        x += speed * math.cos(current_yaw) * dt
        y += speed * math.sin(current_yaw) * dt
        
        current_yaw += active_yaw_rate * dt
        accumulated_yaw += active_yaw_rate * dt
        
        trajectory_locations.append(carla.Location(x=x, y=y, z=z))
        trajectory_coords.append([x, y])
        
    # 2. Convert coordinates to numpy array for batch processing
    trajectory_points = np.array(trajectory_coords)
    
    # 3. Collision / Boundary Check using the KD-Tree
    valid_trajectory_points = query_valid_trajectory(trajectory_points, waypoint_tree, waypoint_array)
    if len(valid_trajectory_points) < len(trajectory_points):
        print(f"Impact detected after {len(valid_trajectory_points)} valid points.")
        return len(valid_trajectory_points), valid_trajectory_points
    else:
        print("No impact detected within the time horizon.")
        return None, trajectory_points  # All points are valid, return full trajectory as carla.Location list

In [8]:
def calculate_rad_ratio(vehicle, waypoint_tree, waypoint_array, steer_intensity=1.0, time_horizon=4.0, dt=0.1):
    """
    Calculates the ratio of distance traveled in a turn vs straight line.
    Useful for normalizing steering input to a 0.0-1.0 range based on how tight the turn is.
    """
    # Simulate a full right turn at max steer
    physics = vehicle.get_physics_control()
    wheelbase_cm = abs(physics.wheels[0].position.x - physics.wheels[2].position.x)
    wheelbase = wheelbase_cm / 100.0
    
    # Calculate the trajectories for right and left turns
    right_turn_length, right_trajectory = predict_trajectory_to_impact(vehicle, wheelbase, waypoint_array, waypoint_tree, turn_direction="right", steer_intensity=steer_intensity, time_horizon=time_horizon, dt=dt)
    left_turn_length, left_trajectory = predict_trajectory_to_impact(vehicle, wheelbase, waypoint_array, waypoint_tree, turn_direction="left", steer_intensity=steer_intensity, time_horizon=time_horizon, dt=dt)

    # Note: special case for handling no impact (infinite turn radius)
    if right_turn_length == -1:
        return None, [], []
    elif right_turn_length == 0 and left_turn_length == 0:
        return None, [], []
    elif right_turn_length is None and left_turn_length is None:
        return 0.5, left_trajectory, right_trajectory # Treat as balanced position
    elif right_turn_length is None:
        right_turn_length = len(right_trajectory) + 1 # Treat as slightly longer than left turn
    elif left_turn_length is None:
        left_turn_length = len(left_trajectory) + 1 # Treat as slightly longer than right turn
    
    ratio = left_turn_length / (left_turn_length + right_turn_length)
    return max(0.0, min(1.0, ratio)), left_trajectory, right_trajectory

In [9]:
current_actors = world.get_actors()

# Filter and destroy lingering vehicles
for ghost_vehicle in current_actors.filter('vehicle.*'):
    ghost_vehicle.destroy()

# Filter and destroy lingering pedestrians (optional, but recommended)
for ghost_walker in current_actors.filter('walker.*'):
    ghost_walker.destroy()

print("Ghost actors cleared.")

# Get blueprints and spawn points
blueprint_library = world.get_blueprint_library()
vehicle_bp = random.choice(blueprint_library.filter('vehicle.*'))
spawn_point = random.choice(carla_map.get_spawn_points())

# Spawn the vehicle and set it to drive automatically
vehicle = world.spawn_actor(vehicle_bp, spawn_point)
vehicle.set_autopilot(True)

print(f"Spawned {vehicle.type_id} at {spawn_point.location}")

Ghost actors cleared.
Spawned vehicle.audi.a2 at Location(x=-446.875336, y=23.061543, z=0.500000)


In [ ]:

spectator = world.get_spectator() 
steer_intensity_setting = 0.4 # Adjust this to make the predicted turn sharper or wider

last_draw_time = 0
draw_interval = 0.1  

try:
    while True:
        world.wait_for_tick() 
        
        # --- SPECTATOR TRACKING ---
        transform = vehicle.get_transform()
        yaw = math.radians(transform.rotation.yaw)
        x_offset = -6.0 * math.cos(yaw)
        y_offset = -6.0 * math.sin(yaw)
        
        camera_location = transform.location + carla.Location(x=x_offset, y=y_offset, z=2.5)
        camera_rotation = carla.Rotation(pitch=-15.0, yaw=transform.rotation.yaw, roll=0.0)
        spectator.set_transform(carla.Transform(camera_location, camera_rotation))

        # --- RAD SLIDER & TRAJECTORY DRAWING ---
        current_time = time.time()
        if current_time - last_draw_time >= draw_interval:
            last_draw_time = current_time 
            
            # 1. Calculate Trajectories to Impact
            ratio, left_traj, right_traj = calculate_rad_ratio(vehicle, waypoint_tree, waypoint_array, steer_intensity=steer_intensity_setting)

            # 3. Draw 3D Text HUD (Floating above the car)
            display_text = f"RAD Slider: {ratio:.2f}" if ratio is not None else "RAD Slider: N/A"
            text_location = transform.location + carla.Location(z=2.0)
            
            world.debug.draw_string(
                text_location, 
                display_text, 
                draw_shadow=True, 
                color=carla.Color(255, 255, 255), 
                life_time=0.15
            )

            # 4. Draw Trajectory Dots
            current_z = transform.location.z
            for loc in left_traj:
                # Assuming loc is a tuple/list like (x, y)
                # Create a carla.Location using loc[0] for x, loc[1] for y, and the adjusted z
                point_3d = carla.Location(x=float(loc[0]), y=float(loc[1]), z=current_z + 0.5)
                world.debug.draw_point(point_3d, size=0.08, color=carla.Color(0, 0, 255), life_time=0.15)
                
            for loc in right_traj:
                point_3d = carla.Location(x=float(loc[0]), y=float(loc[1]), z=current_z + 0.5)
                world.debug.draw_point(point_3d, size=0.08, color=carla.Color(255, 255, 0), life_time=0.15)

except KeyboardInterrupt:
    print("\nScript stopped by user.")
finally:
    if 'vehicle' in locals() and vehicle is not None:
        vehicle.destroy()

In [ ]:
import pygame

pygame.init()
display = pygame.display.set_mode((300, 200))
pygame.display.set_caption("CLICK HERE TO DRIVE")

print("IMPORTANT: Click the new 'CLICK HERE TO DRIVE' window that just opened!")
print("Controls: W (Gas), S (Brake), A/D (Steer), Q (Hold for Reverse)")

# Assuming `world` and `vehicle` are already defined in your environment
carla_map = world.get_map()
vehicle.set_autopilot(False)
control = carla.VehicleControl()
spectator = world.get_spectator() 

steer_intensity_setting = 0.4 # Adjust this to make the predicted turn sharper or wider

last_draw_time = 0
draw_interval = 0.1  

try:
    while True:
        world.wait_for_tick() 

        # --- KEYBOARD CONTROL LOGIC ---
        pygame.event.pump() 
        keys = pygame.key.get_pressed()

        # Throttle / Brake / Steer / Reverse
        control.throttle = min(control.throttle + 0.05, 1.0) if keys[pygame.K_w] else 0.0
        control.brake = min(control.brake + 0.2, 1.0) if keys[pygame.K_s] else 0.0
        
        if keys[pygame.K_a]:
            control.steer = max(control.steer - 0.05, -1.0)
        elif keys[pygame.K_d]:
            control.steer = min(control.steer + 0.05, 1.0)
        else:
            control.steer = 0.0 
            
        control.reverse = keys[pygame.K_q]
        vehicle.apply_control(control)

        # Pygame exit check
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                raise KeyboardInterrupt

        # --- SPECTATOR TRACKING ---
        transform = vehicle.get_transform()
        yaw = math.radians(transform.rotation.yaw)
        x_offset = -6.0 * math.cos(yaw)
        y_offset = -6.0 * math.sin(yaw)
        
        camera_location = transform.location + carla.Location(x=x_offset, y=y_offset, z=2.5)
        camera_rotation = carla.Rotation(pitch=-15.0, yaw=transform.rotation.yaw, roll=0.0)
        spectator.set_transform(carla.Transform(camera_location, camera_rotation))

        # --- RAD SLIDER & TRAJECTORY DRAWING ---
        current_time = time.time()
        if current_time - last_draw_time >= draw_interval:
            last_draw_time = current_time 
            
            # 1. Calculate Trajectories to Impact
            ratio, left_traj, right_traj = calculate_rad_ratio(vehicle, waypoint_tree, waypoint_array, steer_intensity=steer_intensity_setting)

            # 3. Draw 3D Text HUD (Floating above the car)
            display_text = f"RAD Slider: {ratio:.2f}" if ratio is not None else "RAD Slider: N/A"
            text_location = transform.location + carla.Location(z=2.0)
            
            world.debug.draw_string(
                text_location, 
                display_text, 
                draw_shadow=True, 
                color=carla.Color(255, 255, 255), 
                life_time=0.15
            )

            # 4. Draw Trajectory Dots
            current_z = transform.location.z
            for loc in left_traj:
                # Assuming loc is a tuple/list like (x, y)
                # Create a carla.Location using loc[0] for x, loc[1] for y, and the adjusted z
                point_3d = carla.Location(x=float(loc[0]), y=float(loc[1]), z=current_z + 0.5)
                world.debug.draw_point(point_3d, size=0.08, color=carla.Color(0, 0, 255), life_time=0.15)
                
            for loc in right_traj:
                point_3d = carla.Location(x=float(loc[0]), y=float(loc[1]), z=current_z + 0.5)
                world.debug.draw_point(point_3d, size=0.08, color=carla.Color(255, 255, 0), life_time=0.15)

except KeyboardInterrupt:
    print("\nScript stopped by user.")
finally:
    if 'vehicle' in locals() and vehicle is not None:
        vehicle.destroy()
    pygame.quit()
    print("Clean exit.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
IMPORTANT: Click the new 'CLICK HERE TO DRIVE' window that just opened!
Controls: W (Gas), S (Brake), A/D (Steer), Q (Hold for Reverse)
Clean exit.


RuntimeError: time-out of 10000ms while waiting for the simulator, make sure the simulator is ready and connected to localhost:2000